In [6]:
import os
import sys
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))
import scripts.utils as utils
import warnings
%reload_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
df_raw_destination = utils.extract("by_country_of_destination.xls") 
df_raw_destination

,COUNTRY,Unnamed: 1,1981,1982,1983,1984,1985,1986,1987,1988,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,TOTAL
0,ALBANIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,1.0,NaN,NaN,4.0
1,ANDORRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,1.0,NaN,1.0,NaN,2.0,NaN,NaN,5.0
2,ANGOLA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,ANGUILLA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0
4,ANTIGUA AND BARBUDA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,YUGOSLAVIA (Serbia & Montenegro),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,8.0
141,GRAND TOTAL,NaN,48867,53953,42481,41551,45269,49338,56350,58020,...,83640.0,78228.0,80689.0,92998.0,89354.0,79779.0,73719.0,65164.0,15703.0,2515729.0
142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
143,Source: Commission on Filipinos Overseas (CFO),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# clean table
df_clean_destination = utils.clean(df_raw_destination)
df_clean_destination



,COUNTRY,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,TOTAL
0,ALBANIA,0,0,0,0,0,0,0,0,0,...,0,0,3,0,0,0,1,0,0,4.0
1,ANDORRA,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,0,2,0,0,5.0
2,ANGOLA,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,2.0
3,ANGUILLA,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1.0
4,ANTIGUA AND BARBUDA,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,VENEZUELA,1,1,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,4.0
137,VIETNAM,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2.0
138,WAKE ISLAND,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.0
139,YEMEN,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2.0


In [8]:
# build dimension table

unique_destination = (
    df_clean_destination[["COUNTRY"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_destination["Country_ID"] = [f"DST_{i+1:03d}" for i in unique_destination.index]
dim_destination = unique_destination[["Country_ID", "COUNTRY"]].rename(columns={"COUNTRY": "Country"})

dim_destination

,Country_ID,Country
0,DST_001,ALBANIA
1,DST_002,ANDORRA
2,DST_003,ANGOLA
3,DST_004,ANGUILLA
4,DST_005,ANTIGUA AND BARBUDA
...,...,...
136,DST_137,VENEZUELA
137,DST_138,VIETNAM
138,DST_139,WAKE ISLAND
139,DST_140,YEMEN


In [9]:
# make fact table 
fact_table_prep = df_clean_destination.merge(
    dim_destination,
    left_on="COUNTRY",
    right_on="Country",
    how="left"
)
year_cols = [col for col in df_clean_destination.columns if col not in ["COUNTRY", "TOTAL"]]
fact_destination = pd.melt(
    fact_table_prep,
    id_vars=["Country_ID"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Emigrant_Count"

)

fact_destination

,Country_ID,Year,Emigrant_Count
0,DST_001,1981,0
1,DST_002,1981,0
2,DST_003,1981,0
3,DST_004,1981,0
4,DST_005,1981,0
...,...,...,...
5635,DST_137,2020,0
5636,DST_138,2020,0
5637,DST_139,2020,0
5638,DST_140,2020,0


In [10]:
utils.load(dim_destination, "dim_destination.csv")
utils.load(fact_destination, "fact_destination.csv")

downloaded csv file dim_destination.csv
downloaded csv file fact_destination.csv
